# Ejemplo 2 - Agente intermedio

## Propósito didáctico
Este notebook muestra cómo pasar de un chatbot simple a un **agente con herramientas**, capaz de decidir cuándo buscar información o cuándo usar una calculadora.

## Competencias que se trabajan
- Comprender qué es una **tool** o herramienta.
- Distinguir entre responder “solo con el modelo” y responder “con apoyo de herramientas”.
- Construir un agente moderno con `create_agent(...)`.
- Evaluar ventajas, riesgos y límites de un agente más autónomo.

## Idea general
Un modelo conversacional básico solo genera texto.  
Un agente, en cambio, puede **elegir acciones** para resolver mejor una tarea, por ejemplo:
- buscar información;
- realizar operaciones;
- consultar una base de conocimiento;
- llamar a una API.

En este ejemplo usaremos dos herramientas:
1. una búsqueda web simple;
2. una calculadora segura.

## 1. Instalación
Ejecuta esta celda una sola vez.

In [ ]:
#
!pip install -qU langchain langchain-core langchain-groq langchain-community duckduckgo-search

!pip install -U ddgs langchain-community

## 2. Configurar credenciales
Como en el ejemplo básico, la clave se ingresa de forma segura.

In [ ]:
#
import os
import getpass

#os.environ["GROQ_API_KEY"] = "..."

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Ingresa tu Groq API key: ")


## 3. Modelo
En esta sección configuramos el modelo base que razonará sobre la tarea y decidirá cuándo usar herramientas.

### Pregunta de reflexión
¿Por qué no siempre conviene dejar que el modelo responda “de memoria”?

In [ ]:
#
from langchain_groq import ChatGroq

MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=1024,
    max_retries=2,
)


## 4. Herramientas
Aquí definimos las herramientas del agente.

### ¿Qué hace cada una?
- **Búsqueda**: recupera información externa.
- **Calculadora**: resuelve operaciones con más confiabilidad que el modelo generando texto libremente.

### Punto importante
La calculadora está implementada de manera **segura**, evitando el uso directo de `eval(...)`.

In [ ]:
#

from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import ast
import operator as op

search_tool = DuckDuckGoSearchRun()

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
    ast.USub: op.neg,
}


def safe_eval(expr: str):
    def _eval(node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise TypeError("Solo se permiten números.")
        if isinstance(node, ast.BinOp):
            if type(node.op) not in _ALLOWED_OPERATORS:
                raise TypeError("Operación no permitida.")
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            if type(node.op) not in _ALLOWED_OPERATORS:
                raise TypeError("Operación unaria no permitida.")
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.operand))
        raise TypeError("Expresión no permitida.")

    parsed = ast.parse(expr, mode="eval")
    return _eval(parsed.body)


@tool
def busqueda_web(query: str) -> str:
    """Busca información general en internet cuando se necesiten datos externos o recientes."""
    return search_tool.run(query)


@tool
def calculadora(expr: str) -> str:
    """Evalúa expresiones matemáticas seguras, por ejemplo: 23*56 o (10+2)/3."""
    try:
        return str(safe_eval(expr))
    except Exception as e:
        return f"Error en el cálculo: {e}"


## 5. Crear el agente
Este es el paso donde combinamos:
- el modelo,
- las herramientas,
- y las instrucciones generales de comportamiento.

En versiones actuales de LangChain, el patrón recomendado es `create_agent(...)`.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """Eres un asistente útil que puede usar herramientas.
- Usa la búsqueda web sólo cuando la pregunta requiera información externa o reciente.
- Usa la calculadora para operaciones numéricas.
- Si puedes responder sin herramientas, hazlo.
- Responde en español."""

agent = create_agent(
    model=llm,
    tools=[busqueda_web, calculadora],
    system_prompt=SYSTEM_PROMPT,
)


## 6. Función auxiliar para invocar el agente
La función `run_agent(...)` simplifica las pruebas y deja el código más legible.

In [ ]:
def run_agent(user_input: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    last_message = result["messages"][-1]
    return getattr(last_message, "content", str(last_message))


## 7. Pruebas rápidas
Usa estos ejemplos para observar la diferencia entre tipos de tareas:
- una consulta factual reciente;
- un cálculo matemático;
- una pregunta que combine ambas cosas.

### Ejercicio sugerido
Compara el comportamiento del agente con el de un chatbot básico:
- ¿cuándo responde mejor?
- ¿cuándo tarda más?
- ¿en qué situaciones parece más confiable?

In [ ]:
print("=== Agente intermedio ===")
print(run_agent("¿Cuál es la capital de Japón y cuánto es 23 * 56?"))


## 8. La cadena de pensamiento del agente: pensar → actuar → observar

Hasta aquí `run_agent()` nos devuelve solo la última frase. Pero entre tu pregunta y esa frase
pasaron varias cosas, y **ahí es donde vive lo interesante de un agente**: el modelo decidió que
necesitaba una herramienta, eligió cuál, la llamó con ciertos argumentos, leyó lo que regresó, y
recién entonces escribió la respuesta.

Ese ciclo tiene nombre —**ReAct** (*Reasoning + Acting*, Yao et al., 2022)— y es el patrón que
describen las notas del módulo en la sección de agentes. No hay que imaginarlo: `agent.invoke()`
devuelve la traza completa en `result["messages"]`.

| Tipo de mensaje | Qué es en el ciclo |
|---|---|
| `HumanMessage` | Tu pregunta. |
| `AIMessage` con `tool_calls` | **Pensar y actuar**: el modelo decidió qué herramienta usar y con qué argumentos. |
| `ToolMessage` | **Observar**: lo que la herramienta devolvió de vuelta. |
| `AIMessage` final (sin `tool_calls`) | **Responder**: la frase que ves. |

Además, como el modelo es de razonamiento, cada `AIMessage` puede traer su borrador de
deliberación en los metadatos.

In [ ]:
import textwrap


def extraer_razonamiento(mensaje):
    """Devuelve la cadena de pensamiento de un mensaje, o None si no la hay.

    Buscamos en varios lugares a propósito: el nombre del campo cambia entre
    proveedores y entre versiones de langchain-groq.
    """
    extras = {}
    extras.update(getattr(mensaje, "additional_kwargs", None) or {})
    extras.update(getattr(mensaje, "response_metadata", None) or {})

    for clave in ("reasoning_content", "reasoning", "thinking"):
        valor = extras.get(clave)
        if isinstance(valor, str) and valor.strip():
            return valor.strip()

    contenido = getattr(mensaje, "content", None)
    if isinstance(contenido, list):
        piezas = [b.get("reasoning") or b.get("text", "")
                  for b in contenido
                  if isinstance(b, dict) and b.get("type") in ("reasoning", "thinking")]
        piezas = [p for p in piezas if p]
        if piezas:
            return "\n".join(piezas)
    return None


def _bloque(etiqueta, cuerpo, ancho=86):
    print(f"\n[{etiqueta}]")
    print(textwrap.indent(textwrap.fill(str(cuerpo), ancho - 4), "    "))


def run_agent_con_traza(user_input: str) -> str:
    """Como run_agent(), pero imprime el ciclo pensar -> actuar -> observar."""
    resultado = agent.invoke({"messages": [{"role": "user", "content": user_input}]})

    print("=" * 86)
    print(f"PREGUNTA: {user_input}")
    print("=" * 86)

    for mensaje in resultado["messages"]:
        tipo = type(mensaje).__name__

        if tipo == "HumanMessage":
            continue

        if tipo == "ToolMessage":                       # OBSERVAR
            devuelto = str(getattr(mensaje, "content", ""))
            if len(devuelto) > 400:
                devuelto = devuelto[:400] + " […]"
            _bloque(f"OBSERVAR · la herramienta '{getattr(mensaje, 'name', '?')}' devolvió",
                    devuelto)
            continue

        # AIMessage
        pensamiento = extraer_razonamiento(mensaje)
        if pensamiento:
            _bloque("PENSAR · borrador interno del modelo", pensamiento)

        llamadas = getattr(mensaje, "tool_calls", None) or []
        for llamada in llamadas:                        # ACTUAR
            _bloque(f"ACTUAR · llamar a '{llamada['name']}'", f"argumentos: {llamada['args']}")

        contenido = getattr(mensaje, "content", "")
        if not llamadas and isinstance(contenido, str) and contenido.strip():
            _bloque("RESPONDER · esto es lo que ve el usuario", contenido)

    print("\n" + "=" * 86)
    return getattr(resultado["messages"][-1], "content", "")


_ = run_agent_con_traza("¿Cuál es la capital de Japón y cuánto es 23 * 56?")

Vale la pena detenerse en lo que muestra esa traza:

- **El modelo reparte el trabajo.** La capital de Japón la sabe y la contesta de memoria; la
  multiplicación se la pasa a la calculadora. Nadie programó esa decisión: está en el prompt de
  sistema, escrita en español.
- **Los argumentos son del modelo.** El `23*56` que aparece en `ACTUAR` lo escribió él a partir de
  tu pregunta en prosa. Ahí es donde un agente se rompe de formas interesantes: si escribe mal la
  expresión, la herramienta devuelve un error y el modelo tiene que recuperarse.
- **`OBSERVAR` es texto que entra al contexto.** Lo que la herramienta devuelve se convierte en
  más tokens de entrada. Una búsqueda web que regresa mucho texto encarece la llamada y puede
  diluir la pregunta original.

Pruébalo con una pregunta que **obligue a encadenar dos herramientas** —por ejemplo, buscar un
dato numérico y luego operar con él— y verás el ciclo repetirse: pensar, actuar, observar, pensar
otra vez.

> **La misma advertencia de siempre.** El borrador de `PENSAR` es texto generado, no un registro
> fiel de lo que pasó dentro del modelo: sirve para depurar y para entender, no como prueba. Lo
> que sí es evidencia dura son los bloques `ACTUAR` y `OBSERVAR`: ahí sabes con certeza qué
> herramienta se ejecutó, con qué argumentos y qué devolvió.

## 9. Modo interactivo opcional
Este bloque te permite conversar libremente con el agente.

In [ ]:
while True:
    pregunta = input("Tú: ").strip()
    if pregunta.lower() in ["salir", "exit"]:
        print("Agente: ¡Hasta luego!")
        break
    print(f"Agente: {run_agent(pregunta)}")


## 10. Análisis didáctico
### ¿Qué mejora respecto al agente básico?
- Puede apoyarse en herramientas.
- Reduce errores en cálculos sencillos.
- Puede acceder a información más allá del contexto inmediato.

### ¿Qué riesgos aparecen?
- Dependencia de herramientas externas.
- Resultados variables según la calidad de la búsqueda.
- Mayor complejidad del flujo.
- Posibles costos y latencia más altos.

## 11. Actividad sugerida
Agrega una tercera herramienta. Algunas opciones:
- traductor;
- conversor de unidades;
- lector de archivos;
- buscador en una base local.

## 12. Cierre
Este notebook ilustra un cambio importante: pasar de “pedirle cosas a un modelo” a “diseñar un sistema que combine modelo + herramientas”.

## 13. Ideas para mejorarlo más
- Agregar memoria persistente.
- Incorporar herramientas de archivos o RAG.
- Añadir validación estructurada de salidas.
- Implementar streaming y trazabilidad con LangSmith.